# Main plot

In [1]:
# auto reload
%load_ext autoreload
%autoreload 2

In [7]:
from plot import *

In [11]:
problem_list = ["levy10D", "AA33D", "Mazda74D", "levy100D", "Vehicle124", "ackley200D", "trid1000D", "rosenbrock1000D"]
for i, problem in enumerate(problem_list):
    plot_baselines(problem, "straddle", kernel="matern", random=True, legend=i==0, ylabel=(i in [0, 4]), save_path=f"plot_figures/main_{problem}.pdf")

In [12]:
problem_list = ["levy10D", "levy100D"]
ablation_list = ["v_init", "C"]
for problem in problem_list:
    for ablation in ablation_list:
        plot_ablation(problem, ablation, ylabel=problem=="levy10D")

In [8]:
problem_list = ["levy10D", "levy100D"]
for problem in problem_list:
    plot_random(problem, ylabel=problem=="levy10D", legend=problem=="levy10D")

In [36]:
problem_list = ["levy10D", "levy100D"]
for problem in problem_list:
    plot_tr_update(problem, tr_updates=[2, 3, 4, 8], ylabel=problem=="levy10D")

In [37]:
problem_list = ["levy10D", "levy100D"]
for problem in problem_list:
    plot_tr_update(problem, tr_updates=[5, 6, 7], ylabel=problem=="levy10D")

In [11]:
problem_list = ["levy10D", "AA33D", "Mazda74D", "levy100D", "Vehicle124", "ackley200D", "trid1000D", "rosenbrock1000D"]
for i, problem in enumerate(problem_list):
    plot_baselines(problem, "c2lse", kernel="matern", random=True, legend=i==0, ylabel=(i in [0, 4]), save_path=f"plot_figures/c2lse_{problem}.pdf")

In [12]:
problem_list = ["levy10D", "AA33D", "Mazda74D", "levy100D", "Vehicle124", "ackley200D", "trid1000D", "rosenbrock1000D"]
for i, problem in enumerate(problem_list):
    plot_baselines(problem, "TS", kernel="matern", random=True, legend=i==0, ylabel=(i in [0, 4]), save_path=f"plot_figures/TS_{problem}.pdf")

In [13]:
problem_list = ["levy10D", "AA33D", "Mazda74D", "levy100D", "Vehicle124", "ackley200D", "trid1000D", "rosenbrock1000D"]
for i, problem in enumerate(problem_list):
    plot_baselines(problem, "straddle", kernel="rbf", random=True, legend=i==0, ylabel=(i in [0, 4]), save_path=f"plot_figures/rbf_{problem}.pdf")

In [14]:
problem_list = ["levy10D", "AA33D", "Mazda74D", "levy100D", "Vehicle124", "ackley200D", "trid1000D", "rosenbrock1000D"]
for i, problem in enumerate(problem_list):
    plot_baselines(problem, "straddle", kernel="rq", random=True, legend=i==0, ylabel=(i in [0, 4]), save_path=f"plot_figures/rq_{problem}.pdf")

In [15]:
problem_list = ["MC2D", "mishra03"]
for i, problem in enumerate(problem_list):
    plot_baselines(problem, "straddle", kernel="matern", random=True, legend=i==0, ylabel=(i in [0, 4]), save_path=f"plot_figures/main_{problem}.pdf")

# Random sampling

In [ ]:
import numpy as np
try:
    from dataloaders import DataLoader
    from data_store import params
    from utils import fit_gp_model, batch_mean_std
except ImportError as e:
    print(f"Import error: {e}")
    print("Please check if the modules are available and compatible with Python 3.9")
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import torch
import time
device = torch.device("cuda")
dtype = torch.double

data_sizes = {
    "Ackley200D": 1500,
    "Mazda74D": 500,
    "Rosenbrock1000D": 500
}

def get_dataloader(data_name):
    func_params = params[data_name]
    if func_params["grid_size"] is None:
        dataloader = DataLoader(func_params, grid=False)
    else:
        dataloader = DataLoader(func_params)
    return dataloader

def eval(gp, Xtest, ytest, h, epsilon):
    y_mean, _ = batch_mean_std(Xtest, gp)
    lse_preds = (y_mean > h - epsilon).detach().cpu().numpy()
    lse_y = (ytest > h).detach().cpu().numpy()
    prec, rec, f1, _ = precision_recall_fscore_support(
        lse_y, 
        lse_preds, 
        zero_division=0,
        labels=[False, True]
    )
    return f1, prec, rec

def plot_histogram_of_metrics(f1_list, prec_list, rec_list, data_name):
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.hist(f1_list, bins=10, color='blue', alpha=0.7)
    plt.title(f'F1 Score {np.mean(f1_list):.3f} ± {np.std(f1_list):.3f} for {data_name}')
    plt.xlabel('F1 Score')
    plt.ylabel('Frequency')

    plt.subplot(1, 3, 2)
    plt.hist(prec_list, bins=10, color='green', alpha=0.7)
    plt.title(f'Precision {np.mean(prec_list):.3f} ± {np.std(prec_list):.3f} for {data_name}')
    plt.xlabel('Precision')
    plt.ylabel('Frequency')

    plt.subplot(1, 3, 3)
    plt.hist(rec_list, bins=10, color='red', alpha=0.7)
    plt.title(f'Recall {np.mean(rec_list):.3f} ± {np.std(rec_list):.3f} for {data_name}')
    plt.xlabel('Recall')
    plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

def test_stability(data_name):
    dataloader = get_dataloader(data_name)
    func_params = params[data_name]
    epsilon = func_params["epsilon"]
    X, Y = dataloader.gen_observations(data_sizes[data_name])
    gp = fit_gp_model(X, Y, kernel="matern", device=device, dtype=dtype)
    f1_list, prec_list, rec_list = [], [], []
    for i in range(50):
        t0 = time.time()
        Xtest, ytest = dataloader.gen_observations(100000)
        f1, prec, rec = eval(gp, Xtest, ytest, 0, epsilon)
        t1 = time.time()
        print(f"Run {i}: F1={f1:.3f}, Precision={prec:.3f}, Recall={rec:.3f}, Time={t1 - t0:.2f}s")
        f1_list.append(f1[1])
        prec_list.append(prec[1])
        rec_list.append(rec[1])
    plot_histogram_of_metrics(f1_list, prec_list, rec_list, data_name)

Import error: /home/s222509501/.conda/envs/lse/lib/python3.12/site-packages/torch/lib/libtorch_cpu.so: undefined symbol: iJIT_NotifyEvent
Please check if the modules are available and compatible with Python 3.9


ImportError: /home/s222509501/.conda/envs/lse/lib/python3.12/site-packages/torch/lib/libtorch_cpu.so: undefined symbol: iJIT_NotifyEvent

In [6]:
test_stability("Mazda74D")

/home/s222509501/.conda/envs/lse/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/s222509501/.conda/envs/lse/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


NameError: name 'fit_gp_model' is not defined

In [2]:
!conda install linear-operator -y

Channels:
 - defaults
 - conda-forge
 - gpytorch
Platform: linux-64
Solving environment: failed

PackagesNotFoundError: The following packages are not available from current channels:

  - linear-operator

Current channels:

  - defaults
  - https://conda.anaconda.org/conda-forge/linux-64
  - https://conda.anaconda.org/conda-forge/noarch
  - https://conda.anaconda.org/gpytorch/noarch

To search for alternate channels that may provide the conda package you're
looking for, navigate to

    https://anaconda.org

and use the search bar at the top of the page.


